# Raw Data Audit Execution

This notebook shows the operations behind the Stage 1 audit. It uses only the Python standard library and never modifies the original CSV. Saved outputs were generated by executing these cells with `scripts/run_audit_notebook.py`, a sequential Python runner rather than a Jupyter kernel. You may also run the cells in order in Jupyter or VS Code. These are actual rerun outputs, not manually entered results.

## 1. Load the original CSV

Read raw strings, verify the source checksum and inspect a few records. Capture the following code cell and its output together.


In [1]:
from pathlib import Path
import csv, hashlib, io, json, subprocess, sys
from collections import Counter

# Locate the repository from either its root or the notebooks directory.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/raw/source_manifest.json").exists())
manifest = json.loads((ROOT / "data/raw/source_manifest.json").read_text())
raw_path = ROOT / "data/raw" / manifest["file"]
raw_bytes = raw_path.read_bytes()
raw_hash = hashlib.sha256(raw_bytes).hexdigest()
assert raw_hash == manifest["sha256"], "Source checksum mismatch"
reader = csv.DictReader(io.StringIO(raw_bytes.decode("utf-8-sig")))
rows = list(reader)
fields = reader.fieldnames
print("File:", raw_path.name)
print(f"Rows: {len(rows):,} | Columns: {len(fields)} | Source checksum: MATCH")
print("First three records (selected columns):")
for row in rows[:3]:
    print({c: row[c] for c in ["customerID", "tenure", "TotalCharges", "Churn"]})


File: WA_Fn-UseC_-Telco-Customer-Churn.csv
Rows: 7,043 | Columns: 21 | Source checksum: MATCH
First three records (selected columns):
{'customerID': '7590-VHVEG', 'tenure': '1', 'TotalCharges': '29.85', 'Churn': 'No'}
{'customerID': '5575-GNVDE', 'tenure': '34', 'TotalCharges': '1889.5', 'Churn': 'No'}
{'customerID': '3668-QPYBK', 'tenure': '2', 'TotalCharges': '108.15', 'Churn': 'Yes'}


## 2. Investigate TotalCharges blanks

Compare two detection rules instead of assuming that a blank-looking field is zero. Then examine the matching records and check whether the pattern covers every zero-tenure customer.


In [2]:
# Compare literal emptiness with whitespace-aware missing-value detection.
empty_only = [r for r in rows if r["TotalCharges"] == ""]
blank_rows = [r for r in rows if r["TotalCharges"].strip() == ""]
print(f"Empty-string-only check: {len(empty_only)}")
print(f"Whitespace-aware check: {len(blank_rows)}")
print("customerID    tenure  TotalCharges (repr)  Churn")
for r in blank_rows:
    print(f"{r['customerID']:12} {r['tenure']:>6}  {r['TotalCharges']!r:20} {r['Churn']}")
print("Tenure among blank records:", dict(Counter(r["tenure"] for r in blank_rows)))
print("Churn among blank records:", dict(Counter(r["Churn"] for r in blank_rows)))
zero_ids = {r["customerID"] for r in rows if r["tenure"] == "0"}
blank_ids = {r["customerID"] for r in blank_rows}
print("Blank-record IDs equal all zero-tenure IDs:", blank_ids == zero_ids)


Empty-string-only check: 0
Whitespace-aware check: 11
customerID    tenure  TotalCharges (repr)  Churn
4472-LVYGI        0  ' '                  No
3115-CZMZD        0  ' '                  No
5709-LVOEQ        0  ' '                  No
4367-NUYAO        0  ' '                  No
1371-DWPAZ        0  ' '                  No
7644-OMVMY        0  ' '                  No
3213-VVOLG        0  ' '                  No
2520-SGTTA        0  ' '                  No
2923-ARZLG        0  ' '                  No
4075-WKNIU        0  ' '                  No
2775-SEFEE        0  ' '                  No
Tenure among blank records: {'0': 11}
Churn among blank records: {'No': 11}
Blank-record IDs equal all zero-tenure IDs: True


The pattern is consistent with newly joined customers, but does not prove that their accumulated charges equal zero. No imputation or deletion is performed. Treatment options are preserved in `docs/decision_log.md`.

## 3. Rerun the audit and verify preservation

Execute the real audit script twice and compare every output hash. Confirm that the input bytes remain unchanged. Capture the invocation, assertions and output together.


In [3]:
audit_script = ROOT / "scripts/audit_data.py"
output_dir = ROOT / "reports/audit"

def output_hashes():
    return {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
            for p in sorted(output_dir.iterdir()) if p.is_file()}

first = subprocess.run([sys.executable, str(audit_script)], cwd=ROOT,
                       capture_output=True, text=True, check=True)
print(first.stdout, end="")
first_hashes = output_hashes()
subprocess.run([sys.executable, str(audit_script)], cwd=ROOT / "notebooks",
               capture_output=True, text=True, check=True)
assert output_hashes() == first_hashes, "Audit outputs changed on rerun"
assert raw_path.read_bytes() == raw_bytes, "Raw CSV changed"
summary = json.loads((output_dir / "summary.json").read_text())
assert summary["rows"] == len(rows)
assert summary["blank_cells"]["TotalCharges"] == len(blank_rows)
print(f"Verified: {len(first_hashes)} output hashes unchanged on rerun.")
print("Verified: raw CSV unchanged; notebook counts match audit counts.")


AUDIT COMPLETE | 7,043 rows | 21 columns
Duplicates: 0 rows / 0 IDs | Blank TotalCharges: 11
Churn Yes: 1,869 (26.5370%) | Raw unchanged: True
Report: reports/audit/audit.md
Verified: 4 output hashes unchanged on rerun.
Verified: raw CSV unchanged; notebook counts match audit counts.
